# 01 — Dataset Audit

Goal: Inspect the raw structure of all three datasets BEFORE writing any
conversion code. This catches problems early — broken paths, missing annotation
links, wrong file counts — so nothing silently corrupts training data later.

Inputs  : Raw datasets accessed via Drive shortcuts
Outputs : dataset_summary.csv  (outputs/metrics/)
          visdrone_audit_samples.png  (outputs/visualizations/)

Do NOT write any conversion logic here. Audit only.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1: Load config and imports
# ─────────────────────────────────────────────────────────────

import json, os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path
from collections import defaultdict
import cv2

from google.colab import drive
drive.mount('/content/drive')


# ── Re-add repos to path (needed every session — no editable install) ──
CONFIG_PATH = "/content/drive/MyDrive/DLCV_OV_Analytics/configs/config.json"
with open(CONFIG_PATH, "r") as f:
    cfg = json.load(f)

for repo in [cfg["yoloworld_repo"], cfg["sam2_repo"]]:
    if repo not in sys.path:
        sys.path.insert(0, repo)

# ── Unpack paths ──────────────────────────────────────────────
RAW_VISDRONE = cfg["raw_visdrone"]
RAW_COCO     = cfg["raw_coco"]
RAW_YTVIS    = cfg["raw_ytvis"]
VIZ_DIR      = cfg["viz_dir"]
METRICS_DIR  = cfg["metrics_dir"]

os.makedirs(VIZ_DIR,     exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

print("✅ Config loaded.")
print(f"   VisDrone raw : {RAW_VISDRONE}")
print(f"   COCO raw     : {RAW_COCO}")
print(f"   YT-VIS raw   : {RAW_YTVIS}")

Mounted at /content/drive
✅ Config loaded.
   VisDrone raw : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/VisDrone
   COCO raw     : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/COCO
   YT-VIS raw   : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/Youtube VIS


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2: Audit VisDrone DET structure
# Expected: split_folder/images/*.jpg + split_folder/annotations/*.txt
# ─────────────────────────────────────────────────────────────

print("=" * 55)
print("  VisDrone DET — Directory & Annotation Audit")
print("=" * 55)

VISDRONE_CLASSES = {
    0: "ignored_region", 1: "pedestrian", 2: "people",
    3: "bicycle", 4: "car", 5: "van", 6: "truck",
    7: "tricycle", 8: "awning-tricycle", 9: "bus",
    10: "motor", 11: "others",
}

vd_audit = {}

for split in sorted(os.listdir(RAW_VISDRONE)):
    split_path = os.path.join(RAW_VISDRONE, split)
    if not os.path.isdir(split_path):
        continue

    img_dir = os.path.join(split_path, "images")
    ann_dir = os.path.join(split_path, "annotations")

    # Guard: skip if expected subfolders are missing
    if not os.path.isdir(img_dir) or not os.path.isdir(ann_dir):
        print(f"  ⚠️  Skipping {split} — missing images/ or annotations/ subfolder")
        continue

    imgs = sorted(os.listdir(img_dir))
    anns = sorted(os.listdir(ann_dir))

    img_stems = {Path(f).stem for f in imgs if f.endswith(".jpg")}
    ann_stems = {Path(f).stem for f in anns if f.endswith(".txt")}

    matched   = img_stems & ann_stems
    imgs_only = img_stems - ann_stems
    anns_only = ann_stems - img_stems

    class_counts = defaultdict(int)
    total_boxes  = 0
    for ann_file in anns:
        ann_path = os.path.join(ann_dir, ann_file)
        with open(ann_path, "r") as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) < 6:
                    continue
                cat_id = int(parts[5])
                class_counts[cat_id] += 1
                total_boxes += 1

    vd_audit[split] = {
        "n_images"      : len(img_stems),
        "n_annotations" : len(ann_stems),
        "n_matched"     : len(matched),
        "n_imgs_no_ann" : len(imgs_only),
        "n_anns_no_img" : len(anns_only),
        "total_boxes"   : total_boxes,
        "class_dist"    : dict(class_counts),
    }

    print(f"\n  Split: {split}")
    print(f"    Images               : {len(img_stems)}")
    print(f"    Annotations          : {len(ann_stems)}")
    print(f"    Matched pairs        : {len(matched)}")
    print(f"    Images w/o ann       : {len(imgs_only)}")
    print(f"    Anns w/o image       : {len(anns_only)}")
    print(f"    Total bounding boxes : {total_boxes:,}")
    top5 = sorted(class_counts.items(), key=lambda x: -x[1])[:5]
    print(f"    Top 5 classes : {[(VISDRONE_CLASSES.get(k, k), v) for k, v in top5]}")

print("\n✅ VisDrone audit complete.")

  VisDrone DET — Directory & Annotation Audit
  ⚠️  Skipping VisDrone2019-DET-test-challenge — missing images/ or annotations/ subfolder

  Split: VisDrone2019-DET-test-dev
    Images               : 1610
    Annotations          : 1610
    Matched pairs        : 1610
    Images w/o ann       : 0
    Anns w/o image       : 0
    Total bounding boxes : 77,547
    Top 5 classes : [('car', 28074), ('pedestrian', 21006), ('people', 6376), ('motor', 5845), ('van', 5771)]

  Split: VisDrone2019-DET-train
    Images               : 6471
    Annotations          : 6471
    Matched pairs        : 6471
    Images w/o ann       : 0
    Anns w/o image       : 0
    Total bounding boxes : 353,550
    Top 5 classes : [('car', 144867), ('pedestrian', 79337), ('motor', 29647), ('people', 27059), ('van', 24956)]

  Split: VisDrone2019-DET-val
    Images               : 548
    Annotations          : 548
    Matched pairs        : 548
    Images w/o ann       : 0
    Anns w/o image       : 0
    Total b

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3: Audit COCO 2017 structure
# Expected: train2017/, val2017/ images + annotations/*.json
# ─────────────────────────────────────────────────────────────

print("=" * 55)
print("  COCO 2017 — JSON Annotation Audit")
print("=" * 55)

ann_dir    = os.path.join(RAW_COCO, "annotations")
coco_audit = {}

for json_name in ["instances_train2017.json", "instances_val2017.json"]:
    json_path = os.path.join(ann_dir, json_name)

    if not os.path.exists(json_path):
        print(f"  ❌ Missing: {json_path}")
        continue

    print(f"\n  Loading {json_name}...")
    with open(json_path, "r") as f:
        coco_data = json.load(f)

    n_images = len(coco_data.get("images", []))
    n_anns   = len(coco_data.get("annotations", []))
    n_cats   = len(coco_data.get("categories", []))

    id_to_file  = {img["id"]: img["file_name"] for img in coco_data["images"]}
    ann_img_ids = {a["image_id"] for a in coco_data["annotations"]}
    valid_ids   = set(id_to_file.keys())
    orphan_anns = ann_img_ids - valid_ids

    has_seg  = sum(1 for a in coco_data["annotations"] if a.get("segmentation"))
    has_bbox = sum(1 for a in coco_data["annotations"] if a.get("bbox"))
    categories = [c["name"] for c in coco_data["categories"]]

    coco_audit[json_name] = {
        "n_images"     : n_images,
        "n_annotations": n_anns,
        "n_categories" : n_cats,
        "orphan_anns"  : len(orphan_anns),
        "has_seg"      : has_seg,
        "has_bbox"     : has_bbox,
    }

    print(f"    Images       : {n_images:,}")
    print(f"    Annotations  : {n_anns:,}")
    print(f"    Categories   : {n_cats}  →  {categories[:10]}...")
    print(f"    Has bbox     : {has_bbox:,}")
    print(f"    Has seg mask : {has_seg:,}")
    print(f"    Orphan anns  : {len(orphan_anns)}")

        # ── Verify image files exist on disk (safe for large Drive folders) ──
    split_name = "train2017" if "train" in json_name else "val2017"
    img_folder = os.path.join(RAW_COCO, split_name)
    if os.path.isdir(img_folder):
        try:
            # Use a generator + limit count to avoid timing out on 118K files
            gen = (f for f in os.scandir(img_folder) if f.is_file())
            on_disk = sum(1 for _ in zip(gen, range(200000)))  # cap at 200K
            print(f"    Files on disk ({split_name}): ~{on_disk:,} / {n_images:,}")
        except OSError as e:
            print(f"    ⚠️  Could not count files in {split_name}: {e}")
            print(f"       Skipping file count — annotation JSON loaded OK ✅")
    else:
        print(f"    ⚠️  Image folder not found: {img_folder}")

    del coco_data

print("\n✅ COCO audit complete.")

  COCO 2017 — JSON Annotation Audit

  Loading instances_train2017.json...
    Images       : 118,287
    Annotations  : 860,001
    Categories   : 80  →  ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light']...
    Has bbox     : 860,001
    Has seg mask : 860,001
    Orphan anns  : 0
    ⚠️  Could not count files in train2017: [Errno 5] Input/output error: '/content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/COCO/train2017'
       Skipping file count — annotation JSON loaded OK ✅

  Loading instances_val2017.json...
    Images       : 5,000
    Annotations  : 36,781
    Categories   : 80  →  ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light']...
    Has bbox     : 36,781
    Has seg mask : 36,781
    Orphan anns  : 0
    Files on disk (val2017): ~5,000 / 5,000

✅ COCO audit complete.


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4: Audit YouTube-VIS structure
# Actual layout:
#   train/JPEGImages/ + train/instances.json
#   valid/JPEGImages/ + valid/instances.json
#   validation_gt/gt.json, gt_long.json, gt_short.json
# ─────────────────────────────────────────────────────────────

print("=" * 55)
print("  YouTube-VIS — Sequence & Annotation Audit")
print("=" * 55)

ytvis_audit = {}

# ── Audit train and valid splits ──────────────────────────────
for split in ["train", "valid"]:
    split_path  = os.path.join(RAW_YTVIS, split)
    ann_path    = os.path.join(split_path, "instances.json")
    img_root    = os.path.join(split_path, "JPEGImages")

    print(f"\n  Split: {split}")

    if not os.path.isdir(split_path):
        print(f"    ❌ Folder not found: {split_path}")
        continue

    if not os.path.exists(ann_path):
        print(f"    ❌ instances.json not found at: {ann_path}")
        continue

    size_mb = os.path.getsize(ann_path) / 1e6
    print(f"    instances.json size : {size_mb:.1f} MB")

    try:
        with open(ann_path, "r") as f:
            ytvis_data = json.load(f)
    except json.JSONDecodeError as e:
        print(f"    ❌ JSONDecodeError: {e} — file corrupted, re-download needed.")
        continue

    n_videos   = len(ytvis_data.get("videos", []))
    n_anns     = len(ytvis_data.get("annotations", []))
    n_cats     = len(ytvis_data.get("categories", []))
    categories = [c["name"] for c in ytvis_data.get("categories", [])]

    frame_counts = [v.get("length", 0) for v in ytvis_data.get("videos", [])]
    avg_frames   = np.mean(frame_counts) if frame_counts else 0
    max_frames   = max(frame_counts)     if frame_counts else 0
    min_frames   = min(frame_counts)     if frame_counts else 0

    # Count video folders on disk safely
    n_folders_on_disk = 0
    if os.path.isdir(img_root):
        try:
            n_folders_on_disk = len(os.listdir(img_root))
        except OSError as e:
            print(f"    ⚠️  Could not count JPEGImages folders: {e}")
            n_folders_on_disk = -1
    else:
        print(f"    ⚠️  JPEGImages folder not found: {img_root}")

    ytvis_audit[split] = {
        "n_videos"       : n_videos,
        "n_anns"         : n_anns,
        "n_categories"   : n_cats,
        "avg_frames"     : round(avg_frames, 1),
        "frame_range"    : (min_frames, max_frames),
        "folders_on_disk": n_folders_on_disk,
    }

    print(f"    Videos               : {n_videos}")
    print(f"    Annotations          : {n_anns}")
    print(f"    Categories ({n_cats})   : {categories[:10]}...")
    print(f"    Avg frames/video     : {avg_frames:.1f}")
    print(f"    Frame range          : {min_frames} – {max_frames}")
    if n_folders_on_disk >= 0:
        status = "✅" if n_folders_on_disk == n_videos else "⚠️ "
        print(f"    Video folders on disk: {status} {n_folders_on_disk} / {n_videos}")

    del ytvis_data

# ── Audit validation_gt JSONs ─────────────────────────────────
print("\n  validation_gt/ JSONs:")
vgt_path = os.path.join(RAW_YTVIS, "validation_gt")
for fname in ["gt.json", "gt_long.json", "gt_short.json"]:
    fpath = os.path.join(vgt_path, fname)
    if not os.path.exists(fpath):
        print(f"    ⚠️  Not found: {fname}")
        continue
    size_mb = os.path.getsize(fpath) / 1e6
    try:
        with open(fpath, "r") as f:
            data = json.load(f)
        n_vids = len(data.get("videos", []))
        n_anns = len(data.get("annotations", []))
        print(f"    ✅ {fname} ({size_mb:.1f} MB) — {n_vids} videos, {n_anns} annotations")
        del data
    except json.JSONDecodeError as e:
        print(f"    ❌ {fname} corrupted: {e}")

# ── Note on test.json ─────────────────────────────────────────
test_path = os.path.join(RAW_YTVIS, "test.json")
if os.path.exists(test_path):
    size_mb = os.path.getsize(test_path) / 1e6
    print(f"\n  test.json ({size_mb:.1f} MB) — skipped (known corrupted, not needed for training)")

print("\n✅ YouTube-VIS audit complete.")

  YouTube-VIS — Sequence & Annotation Audit

  Split: train
    instances.json size : 648.8 MB
    Videos               : 2985
    Annotations          : 6283
    Categories (40)   : ['airplane', 'bear', 'bird', 'boat', 'car', 'cat', 'cow', 'deer', 'dog', 'duck']...
    Avg frames/video     : 30.2
    Frame range          : 5 – 72
    Video folders on disk: ✅ 2985 / 2985

  Split: valid
    instances.json size : 0.5 MB
    Videos               : 492
    Annotations          : 0
    Categories (40)   : ['airplane', 'bear', 'bird', 'boat', 'car', 'cat', 'cow', 'deer', 'dog', 'duck']...
    Avg frames/video     : 33.8
    Frame range          : 5 – 84
    Video folders on disk: ✅ 492 / 492

  validation_gt/ JSONs:
    ✅ gt.json (121.0 MB) — 492 videos, 1157 annotations
    ✅ gt_long.json (26.0 MB) — 71 videos, 259 annotations
    ✅ gt_short.json (95.0 MB) — 421 videos, 898 annotations

  test.json (1.5 MB) — skipped (known corrupted, not needed for training)

✅ YouTube-VIS audit complete.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5: Visualize sample annotated VisDrone images
# ─────────────────────────────────────────────────────────────

VISDRONE_CLASSES = {
    0: "ignored", 1: "pedestrian", 2: "people", 3: "bicycle",
    4: "car", 5: "van", 6: "truck", 7: "tricycle",
    8: "awning-tricycle", 9: "bus", 10: "motor", 11: "others",
}

COLOR_MAP = {
    1: (255, 80, 80),  2: (255, 160, 80), 3: (80, 180, 80),
    4: (80, 120, 255), 5: (180, 80, 255), 6: (255, 220, 60),
    7: (80, 220, 220), 8: (200, 120, 80), 9: (255, 100, 180),
    10:(100, 200, 100),
}

def parse_visdrone_annotation(ann_path):
    boxes = []
    with open(ann_path, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            x, y, w, h  = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
            category_id  = int(parts[5])
            if category_id == 0 or w <= 0 or h <= 0:
                continue
            boxes.append({"x": x, "y": y, "w": w, "h": h, "cat": category_id})
    return boxes

# Dynamically find the train split folder
available_splits = [d for d in os.listdir(RAW_VISDRONE)
                    if os.path.isdir(os.path.join(RAW_VISDRONE, d))]
train_splits = [d for d in available_splits if "train" in d.lower()]
if not train_splits:
    raise FileNotFoundError(f"No train split found in {RAW_VISDRONE}. Found: {available_splits}")
train_split = os.path.join(RAW_VISDRONE, train_splits[0])
print(f"Using train split: {train_split}")

img_dir = os.path.join(train_split, "images")
ann_dir = os.path.join(train_split, "annotations")

sample_imgs = sorted(os.listdir(img_dir))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, img_file in enumerate(sample_imgs):
    img_path = os.path.join(img_dir, img_file)
    ann_path = os.path.join(ann_dir, Path(img_file).stem + ".txt")

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    boxes = parse_visdrone_annotation(ann_path) if os.path.exists(ann_path) else []
    for b in boxes:
        color = COLOR_MAP.get(b["cat"], (200, 200, 200))
        cv2.rectangle(img, (b["x"], b["y"]),
                      (b["x"] + b["w"], b["y"] + b["h"]), color, 1)
        label = VISDRONE_CLASSES.get(b["cat"], str(b["cat"]))
        cv2.putText(img, label, (b["x"], max(0, b["y"] - 3)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)

    axes[i].imshow(img)
    axes[i].set_title(f"{img_file}\n{len(boxes)} boxes", fontsize=9)
    axes[i].axis("off")

plt.suptitle("VisDrone DET — Sample Annotated Images", fontsize=14)
plt.tight_layout()
save_path = os.path.join(VIZ_DIR, "visdrone_audit_samples.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {save_path}")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6: Save dataset audit summary to CSV
# ─────────────────────────────────────────────────────────────

rows = []

for split, info in vd_audit.items():
    rows.append({
        "dataset"       : "VisDrone",
        "split"         : split,
        "n_images"      : info["n_images"],
        "n_annotations" : info["n_annotations"],
        "n_matched"     : info["n_matched"],
        "total_boxes"   : info["total_boxes"],
        "orphan_imgs"   : info["n_imgs_no_ann"],
        "orphan_anns"   : info["n_anns_no_img"],
        "has_masks"     : False,
        "is_video"      : False,
    })

for json_name, info in coco_audit.items():
    split = "train" if "train" in json_name else "val"
    rows.append({
        "dataset"       : "COCO2017",
        "split"         : split,
        "n_images"      : info["n_images"],
        "n_annotations" : info["n_annotations"],
        "n_matched"     : info["n_images"],
        "total_boxes"   : info["has_bbox"],
        "orphan_imgs"   : 0,
        "orphan_anns"   : info["orphan_anns"],
        "has_masks"     : info["has_seg"] > 0,
        "is_video"      : False,
    })

for json_name, info in ytvis_audit.items():
    split = json_name.replace(".json", "")
    rows.append({
        "dataset"       : "YouTubeVIS",
        "split"         : split,
        "n_images"      : info["n_videos"],
        "n_annotations" : info["n_anns"],
        "n_matched"     : info["folders_on_disk"],
        "total_boxes"   : 0,
        "orphan_imgs"   : info["n_videos"] - info["folders_on_disk"],
        "orphan_anns"   : 0,
        "has_masks"     : True,
        "is_video"      : True,
    })

df = pd.DataFrame(rows)
summary_path = os.path.join(METRICS_DIR, "dataset_summary.csv")
df.to_csv(summary_path, index=False)

print("=" * 60)
print("  DATASET AUDIT SUMMARY")
print("=" * 60)
print(df.to_string(index=False))
print(f"\n✅ Saved: {summary_path}")

  DATASET AUDIT SUMMARY
   dataset                     split  n_images  n_annotations  n_matched  total_boxes  orphan_imgs  orphan_anns  has_masks  is_video
  VisDrone VisDrone2019-DET-test-dev      1610           1610       1610        77547            0            0      False     False
  VisDrone    VisDrone2019-DET-train      6471           6471       6471       353550            0            0      False     False
  VisDrone      VisDrone2019-DET-val       548            548        548        40169            0            0      False     False
  COCO2017                     train    118287         860001     118287       860001            0            0       True     False
  COCO2017                       val      5000          36781       5000        36781            0            0       True     False
YouTubeVIS                     train      2985           6283       2985            0            0            0       True      True
YouTubeVIS                     valid       49